In [1]:
import GradientGang.Pipeline.Optimizer.OptunaOptimizer
from GradientGang.Pipeline.Optimizer.OptunaOptimizer import OptunaOptimizer

In [2]:
data_params_s = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.1,
    'shuffle': True,
}
data_params_d = {"stage": "fit", "includeTestInTrain": False}
architectureDirect = {
    "arch_type": "direct",
    "LearningRate": 0.001,
    "Patience": 5,
    #"ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
    "RegularizationWeight": 0.1,
    "EncoderParams": {
        "activation_function": "GELU",
        "layer_type": [
            {
                "name": "GRU",
                "params": {
                    "input_size": 34,
                    "hidden_size": 128,
                    "num_layers": 2,
                    "bias": True,
                    "batch_first": True,
                    "dropout": 0.2,
                    "bidirectional": True  # Output will be hidden_size * 2 = 256
                }
            },
            {
                "name": "Linear",
                "params": {
                    "in_features": 256,  # 128 * 2 because bidirectional=True
                    "out_features": 128,
                    "bias": True,
                }
            }
        ]
    },
    "GlobalFFEncoderParams": {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 1,
                    "out_features": 1,
                    "bias": True,
                }
            },
        ]
    },
    "FeedForwardParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 129,  # 128 (encoder) + 1 (global feature)
                    "out_features": 64,
                    "bias": True,
                }
            },
            {
                "name": "Linear",
                "params": {
                    "in_features": 64,
                    "out_features": 32,
                    "bias": True,
                }
            }
        ]
    },
    "OutputDim": 3  # Number of classes
}
hyper_arch = {
    "hyper_num_layers":{
        "name": "num_layers",
        "type": "int",
        "paths": [["EncoderParams", "layer_type", 0, "params"]],
        "opts": {"low": 1, "high": 5}
    },
    "dropout": {
        "name": "dropout",
        "type": "float",
        "paths": [["EncoderParams", "layer_type", 0, "params"]],
        "opts": {"low": 0.1, "high": 0.9}
    },
    "hyper_batch_first":{
        "name": "batch_first",
        "type": "categ",
        "paths": [["EncoderParams", "layer_type", 0, "params"]],
        "opts": {"choices": [True, False]}
    }
}
hyper_data = {
    "e": {
        "name": "stage",
        "type": "value",
        "paths": [[]],
        "opts": {"value": "fit"}
    },
    "includeTestInTrain": {
        "name": "includeTestInTrain", 
        "type": "categ",
        "paths": [[]],
        "opts": {"choices": [True, False]}
    }
}

In [3]:
params = {"dataloader": data_params_s, "hyper_dataloader": hyper_data, "arch": architectureDirect, "hyper_arch": hyper_arch}

In [4]:
optimizer = OptunaOptimizer(params)

In [6]:
optimizer.optimize(1)

[I 2025-11-11 18:54:30,130] A new study created in memory with name: no-name-e771cd9f-4532-4664-bf0a-dcb7065b5581
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name             | Type              | Params | Mode 
---------------------------------------------------------------
0 | encoder          | Encoder           | 455 K  | train
1 | globalff_encoder | FeedForward       | 2      | train
2 | feedforward      | FeedForward       | 10.5 K | train
3 | val_f1           | MulticlassF1Score | 0      | train
---------------------------------------------------------------
465 K     Trainable params
0         Non-trainable params
465 K     Total params
1.863     Total estimated model params size (MB)
19        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\teopa\Documents\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\teopa\Documents\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\teopa\Documents\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (19) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.
[I 2025-11-11 18:55:01,595] Trial 0 finished with value: 0.7493112683296204 and parameters: {'includeTestInTrain': False, 'num_layers': 4, 'dropout': 0.5359065463975176, 'batch_first': False}. Best is trial 0 with value: 0.7493112683296204.
